# City Senses · city series template

**How to use:** File → Save As → `dayNN_prompt_city.ipynb`, change the settings, Run All.

Same six cities every time, all from OpenStreetMap. The only thing you change is **what** you pull (the OSM tags).
The map opens on the first city and has buttons to fly between them.
This template is set up as an example: **cafés in six cities**.

## 1 · Settings

In [ ]:
DAY      = 1
PROMPT   = "Points"
STUDY    = "City"
SLUG     = "cafes-city"
NOTEBOOK = "_template_city.ipynb"

TITLE    = "Six Cities, One Coffee Habit"
SUBTITLE = "Every café OpenStreetMap knows about"
STORY    = "Which of these cities runs on coffee? Use the buttons to fly between them."
CREDITS  = ["osm"]

CITIES   = ["nyc", "philly", "dc", "sf", "austin", "seattle"]   # see mapkit/city.py
TAGS     = {"amenity": "cafe"}     # what to pull, ideas in mapkit/city.py
NAME     = "cafes"                 # short name for the cached files
COLOR_BY = None                    # e.g. "cuisine" to color by a column, None = one color
TOOL     = "folium"                # "folium" (with city buttons) | "static"

## 2 · Load

In [ ]:
import mapkit as mk
from mapkit import city, publish

pts = city.fetch(TAGS, name=NAME, cities=CITIES)   # first run asks OSM, then it's cached in data/raw
print(f"{len(pts):,} {NAME}")
pts["city_label"].value_counts()

## 3 · Analysis

In [ ]:
stats = city.per_km2(pts, CITIES)       # count per km² of city land, great for the caption
if COLOR_BY:
    pts["color_group"] = city.top_values(pts, COLOR_BY, n=6)
stats

## 4 · Map

In [ ]:
layer = mk.Layer(pts, TITLE.split(",")[0] if not COLOR_BY else COLOR_BY.title(),
                 column="color_group" if COLOR_BY else None, radius=3.5, opacity=0.85,
                 popup={"name": "Name", "city_label": "City"})
first = city.CITIES[CITIES[0]]

if TOOL == "folium":
    m = mk.interactive([layer], day=DAY, prompt=PROMPT, title=TITLE, subtitle=SUBTITLE, story=STORY,
                       sources=mk.style.credits(*CREDITS), series="city", switcher={c: None for c in CITIES},
                       start=(*first["center"], first["zoom"]))
else:
    m = mk.static([layer], day=DAY, prompt=PROMPT, title=TITLE,
                  subtitle=SUBTITLE, sources=mk.style.credits(*CREDITS), series="city")
m

## 5 · Export

In [ ]:
publish.save(m, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE, subtitle=SUBTITLE, series="city", study=STUDY,
             tool="Python + Folium" if TOOL == "folium" else "Python", notebook=NOTEBOOK,
             credits=mk.style.credits(*CREDITS, full=True))